# Lab 12 — Real Robot Localization

Same Bayes filter as `sim_demo.ipynb`, but the BLE round-trip goes to a real Artemis instead of `SimRobot`. Each `DriveUpdate` returns the latest cached ToF samples plus the yaw at sample time, and we feed that straight into `Localization2D.predict / update`.

Before running:
1. Power the robot, place it at the known start pose with a known initial heading.
2. The MAC address + UUIDs come from `ble_robot_1.4/main_python/ble/connection.yaml`.
3. The IMU yaw will be snapped to `START_THETA_DEG` via `SetHeading` once connected, so the gyro-trusted assumption holds in the world frame.

## 1. Setup

In [ ]:
%matplotlib inline
import sys, pathlib

SIM_DIR = pathlib.Path.cwd().parent          # labs/lab12/sim
BLE_DIR = SIM_DIR.parent / 'ble_robot_1.4'   # labs/lab12/ble_robot_1.4
for p in (SIM_DIR, BLE_DIR):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import asyncio, math
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

from world import load_world
from localization2d import Localization2D
from controller import WaypointController, BeliefPlot, filter_step, step_once
from real_robot import RealRobot
from paths import LAB_PATH_M

from main_python.ble.connection import BLEConnection
from main_python.commands import SetHeading


In [ ]:
PATH = LAB_PATH_M
START_THETA_DEG  = 45.0
BLE_TIMEOUT_S    = 5.0
# Controller + filter knobs live in config/world.yaml.

## 2. Build localization, controller, and plot

In [ ]:
map_lines, bounds, loc_params, _, ctl_kwargs = load_world()
loc = Localization2D(map_lines, bounds, params=loc_params)
loc.init_at(PATH[0][0], PATH[0][1], sigma_m=0.10)
ctl = WaypointController(path=PATH, **ctl_kwargs)
plot = BeliefPlot(loc, map_lines, path=PATH)
plot.update(None, type('R', (), {'tof1_dist_mm': 0, 'tof2_dist_mm': 0,
                                  'tof1_yaw_deg': 0, 'tof2_yaw_deg': 0})(),
            title='initial belief')
plt.show()

## 3. Connect and seed the gyro

Establishes the BLE link and snaps `imu::yaw` on the robot to `START_THETA_DEG`, so all subsequent `tof_yaw_deg` fields land in the world frame.

In [ ]:
conn = BLEConnection()
await conn.connect()
await conn.execute(SetHeading(START_THETA_DEG), timeout=BLE_TIMEOUT_S)
robot = RealRobot(conn, timeout_s=BLE_TIMEOUT_S)
print('connected, heading set to', START_THETA_DEG, 'deg')

## 4. Stationary sanity check

Hold the robot still and run a few `DriveUpdate(speed=0, heading=0)` calls. The belief should sharpen around the true pose without driving anywhere; if it doesn't, the start pose, heading, or sensor sigma is off before you turn the motors on.

In [ ]:
last_t_us = 0
for i in range(20):
    resp = await robot.update(target_speed=0.0, target_heading=START_THETA_DEG)
    dt_s = max(1e-3, (resp.current_us - last_t_us) * 1e-6) if last_t_us else 0.1
    last_t_us = resp.current_us
    filter_step(loc, resp, speed_pwm=0.0, dt_s=dt_s)
    bx, by = loc.mean_xy()
    print(f'i={i:2d}  bel=({bx:+.3f},{by:+.3f})  '
          f'tof1={resp.tof1_dist_mm}mm@{resp.tof1_yaw_deg:+.1f}  '
          f'tof2={resp.tof2_dist_mm}mm@{resp.tof2_yaw_deg:+.1f}')
    plot.update(None, resp, title=f'stationary  i={i}')
    clear_output(wait=True)
    display(plot.fig)

## 5. Drive the path

Closed-loop: the controller picks `(speed, heading)` from the current belief mean, the robot drives one BLE round-trip, the filter updates, and we redraw. Stop with `Kernel → Interrupt` (the `finally` issues a `DriveUpdate(0, 0)` so the robot's drive timeout will zero the motors).

In [ ]:
last_t_us = 0
last_yaw = START_THETA_DEG
step = 0
try:
    while not ctl.done:
        resp, last_t_us = await step_once(
            robot, loc, ctl,
            last_t_us=last_t_us, last_yaw_deg=last_yaw,
        )
        last_yaw = resp.yaw_deg
        step += 1
        plot.update(None, resp,
                    title=f'step {step}  wp {ctl.idx}/{len(ctl.path)-1}  '
                          f'yaw={resp.yaw_deg:+.1f}deg  flip={ctl.flip}')
        clear_output(wait=True)
        display(plot.fig)
finally:
    try:
        await robot.update(target_speed=0.0, target_heading=last_yaw)
    except Exception as e:
        print('shutdown update failed:', e)
    print(f'finished after {step} steps. final belief mean = {loc.mean_xy()}')

## 6. Disconnect

In [ ]:
await conn.disconnect()
print('disconnected')